# Week 4：LDA 主题模型
Latent Dirichlet Allocation，潜在狄利克雷分配

K-means 为每篇文章分配了一个簇编号，LDA 则把一篇文章表示为多个主题的混合。

例如，一篇介绍太空图像处理的文章，可能同时涉及太空科学和计算机图形学。

模型估计的主题比例，合计为 1。

我们需要区分两个概念：
| 概念 | 回答的问题 |
|---|---|
| 文档—主题分布 | 一篇文章由哪些主题组成，各占多少？ |
| 主题—词分布 | 一个主题更倾向于使用哪些词？ |

模型先学习词的分布，主题名称由我们阅读关键词后概括。
## 实验目的

使用与文本表示实验相同的 20 Newsgroups 四类别子集，
学习 LDA 主题模型的基本流程。

实验任务：
1. 构建词频矩阵。
2. 分别训练包含 5、10、15 个主题的 LDA 模型。
3. 输出每个主题中权重最高的 10 个词。
4. 观察主题含义及不同主题数带来的变化。
5. 检查文档的主题分布。

需要回答：
- 文档聚类与主题发现有什么区别？
- 一篇文档能否同时包含多个主题？
- 没有类别标签时，主题模型为什么有用？

真实类别标签不参与 LDA 训练。

In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

RANDOM_SEED = 42

categories = [
    "comp.graphics",
    "sci.space",
    "alt.atheism",
    "talk.religion.misc",
]

newsgroups = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=RANDOM_SEED,
)

text_df = pd.DataFrame({
    "text": newsgroups.data,
    "label": newsgroups.target,
})

label_to_category = dict(enumerate(newsgroups.target_names))
text_df["category"] = text_df["label"].map(label_to_category)

empty_mask = text_df["text"].str.strip().eq("")

clean_text_df = (
    text_df.loc[~empty_mask]
    .copy()
    .reset_index(names="original_index")
)

documents = clean_text_df["text"].tolist()
true_labels = clean_text_df["label"].to_numpy()

assert len(documents) == len(true_labels)
assert all(document.strip() for document in documents)

print("Original document count:", len(text_df))
print("Removed empty documents:", int(empty_mask.sum()))
print("Remaining document count:", len(documents))

print("\nDocuments per category:")
print(clean_text_df["category"].value_counts().sort_index())

Original document count: 3387
Removed empty documents: 92
Remaining document count: 3295

Documents per category:
category
alt.atheism           779
comp.graphics         955
sci.space             955
talk.religion.misc    606
Name: count, dtype: int64


## 1. 构建词频矩阵
什么是词频？

词频矩阵直接记录这些出现次数。TF-IDF 则会结合词在整个文档集合中的分布，对次数进行加权。

使用 CountVectorizer 将文章转换为词项计数矩阵：
- 每一行对应一篇文章。
- 每一列对应词表中的一个词项。
- 每个元素记录该词项在该文章中的出现次数。

参数设置：
- 转换为小写。
- 过滤内置英文停用词。
- 保留至少出现在 2 篇文档中的词。
- 过滤出现在超过 95% 文档中的词。
- 最多保留 5000 个词项，按整个文档集合中的总出现次数选择。

保留原始计数，不进行 TF-IDF 加权或 L2 归一化。

后续 5、10、15 个主题的实验将使用同一个词频矩阵。

In [2]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=2,
    max_df=0.95,
    max_features=5000,
)

X_counts = count_vectorizer.fit_transform(documents)
count_feature_names = count_vectorizer.get_feature_names_out()

retained_word_counts = np.asarray(X_counts.sum(axis=1)).ravel()
zero_count_mask = retained_word_counts == 0

print("Count matrix shape:", X_counts.shape)
print("Number of documents:", X_counts.shape[0])
print("Vocabulary size:", len(count_feature_names))
print("Matrix data type:", X_counts.dtype)
print("Zero-count documents:", int(zero_count_mask.sum()))
print("Retained word occurrences in first document:", int(retained_word_counts[0]))

Count matrix shape: (3295, 5000)
Number of documents: 3295
Vocabulary size: 5000
Matrix data type: int64
Zero-count documents: 15
Retained word occurrences in first document: 32


## 2. 训练 5 主题 LDA 模型

输入为形状 (3295, 5000) 的词频矩阵。

参数设置：
- 主题数：5
- 学习方式：batch
- 最大训练轮数：20
- 文档—主题先验参数：0.1
- 主题—词先验参数：0.1
- 随机种子：42
- 并行任务数：1

保留 15 篇全零计数文档，并记录这一局限。
这些文档缺少词频证据，后续不用于解释具体文章的主题。

后续 10、15 主题实验沿用同一个词频矩阵和上述其他参数。

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from threadpoolctl import threadpool_limits

lda_5 = LatentDirichletAllocation(
    n_components=5,#让模型学习5个主题
    learning_method="batch",
    max_iter=20,
    doc_topic_prior=0.1,#一篇文档中的主题比例
    topic_word_prior=0.1,#一个主题中的词语分布
    evaluate_every=-1,
    random_state=RANDOM_SEED,
    n_jobs=1,
    verbose=1,
)

print("Training LDA with 5 topics...")

with threadpool_limits(limits=1):
    doc_topic_5 = lda_5.fit_transform(X_counts)

print("LDA training completed.")
print("Training iterations:", lda_5.n_iter_)
print("Document-topic matrix shape:", doc_topic_5.shape)
print("Topic-word parameter shape:", lda_5.components_.shape)
print("First 5 document-topic row sums:", doc_topic_5[:5].sum(axis=1))

Training LDA with 5 topics...
iteration: 1 of max_iter: 20
iteration: 2 of max_iter: 20
iteration: 3 of max_iter: 20
iteration: 4 of max_iter: 20
iteration: 5 of max_iter: 20
iteration: 6 of max_iter: 20
iteration: 7 of max_iter: 20
iteration: 8 of max_iter: 20
iteration: 9 of max_iter: 20
iteration: 10 of max_iter: 20
iteration: 11 of max_iter: 20
iteration: 12 of max_iter: 20
iteration: 13 of max_iter: 20
iteration: 14 of max_iter: 20
iteration: 15 of max_iter: 20
iteration: 16 of max_iter: 20
iteration: 17 of max_iter: 20
iteration: 18 of max_iter: 20
iteration: 19 of max_iter: 20
iteration: 20 of max_iter: 20
LDA training completed.
Training iterations: 20
Document-topic matrix shape: (3295, 5)
Topic-word parameter shape: (5, 5000)
First 5 document-topic row sums: [1. 1. 1. 1. 1.]


两个 prior：为概率分布设置先验
| 参数 | 作用对象 |
|---|---|
| `doc_topic_prior` | 一篇文档中的主题比例 |
| `topic_word_prior` | 一个主题中的词语分布 |

可以先把“先验”理解为：模型结合当前词频证据进行估计时，使用的一种初始分布偏好。

这里的 0.1 不是“每个主题占 10%”，也不是学习率。我们显式固定它们，是因为留空时，它们的默认值会随主题数变化。

## 3. 查看 5 主题模型的关键词

从每个主题对应的词项参数中，
提取权重最高的 10 个词，并按权重从高到低排列。

这些词用于帮助解释主题含义。
主题编号不代表真实类别，主题名称需要根据关键词及相关文档概括。

In [4]:
def get_topic_words(model, feature_names, n_top_words=10):
    rows = []

    for topic_index, topic_weights in enumerate(model.components_):
        top_indices = topic_weights.argsort()[-n_top_words:][::-1]
        top_words = feature_names[top_indices]

        rows.append({
            "Topic": topic_index,
            "Top words": ", ".join(top_words),
        })

    return pd.DataFrame(rows)


topics_5_df = get_topic_words(lda_5, count_feature_names)

with pd.option_context("display.max_colwidth", None):
    display(topics_5_df)

,Topic,Top words
0,0,"space, nasa, earth, launch, orbit, shuttle, mission, moon, solar, satellite"
1,1,"god, people, jesus, bible, believe, does, religion, say, know, don"
2,2,"just, don, like, think, people, know, does, ve, time, way"
3,3,"edu, jpeg, graphics, file, image, gif, ftp, files, images, format"
4,4,"image, data, software, program, processing, computer, images, using, line, available"


### 5 主题模型的关键词观察

| 主题编号 | 暂定含义 | 观察 |
|---|---|---|
| 0 | 太空与航天 | NASA、发射、轨道、航天飞机等关键词具有较明确的领域含义 |
| 1 | 宗教与信仰讨论 | 上帝、耶稣、圣经、信仰等词具有一致的讨论方向 |
| 2 | 通用讨论语言 | 主要由常见讨论用词构成，暂时难以概括具体主题 |
| 3 | 图像格式与文件交换 | JPEG、GIF、文件、FTP、格式等词较集中 |
| 4 | 图像处理与软件工具 | 图像处理、程序、软件等词体现一定技术含义 |

主题 3 与主题 4 存在内容重叠，但关键词侧重点不同。
这表明同一个真实类别可能包含多个讨论主题。

目前没有仅凭关键词识别出一个明确对应“无神论”的独立主题，
不能因此认定无神论文章没有被建模。
这些文章可能混合了宗教讨论及其他主题，需要查看文档的主题比例。

主题名称是人工解释，不是模型输出的真实类别。
以上判断仅基于前十个词，后续需要结合文档内容检查。

In [ ]:
#看一篇文章的主题比例
example_index = int(np.flatnonzero(~zero_count_mask)[0])#选择一篇有词频证据的文章

print("Document position:", example_index)
print(
    "Original document index:",
    clean_text_df.iloc[example_index]["original_index"],
)

print("\nDocument preview:")
print(documents[example_index][:1000])

example_topic_df = pd.DataFrame({
    "Topic": np.arange(lda_5.n_components),
    "Proportion": doc_topic_5[example_index],#取出这篇文章对应的一行
})

example_topic_df = example_topic_df.sort_values(
    "Proportion",
    ascending=False,
)#按比例从高到低显示

print("\nDocument-topic proportions:")
display(example_topic_df.round(4))

print("Proportion sum:", doc_topic_5[example_index].sum())

Document position: 0
Original document index: 0

Document preview:
My point is that you set up your views as the only way to believe.  Saying 
that all eveil in this world is caused by atheism is ridiculous and 
counterproductive to dialogue in this newsgroups.  I see in your posts a 
spirit of condemnation of the atheists in this newsgroup bacause they don'
t believe exactly as you do.  If you're here to try to convert the atheists 
here, you're failing miserably.  Who wants to be in position of constantly 
defending themselves agaist insulting attacks, like you seem to like to do?!
I'm sorry you're so blind that you didn't get the messgae in the quote, 
everyone else has seemed to.

Document-topic proportions:


,Topic,Proportion
1,1,0.7726
2,2,0.2182
3,3,0.0031
4,4,0.0031
0,0,0.0031


Proportion sum: 1.0


这篇文章的结果

| 主题 | 比例 | 结合原文的解释 |
|---|---:|---|
| 1：宗教与信仰讨论 | 77.26% | 文章讨论信仰、无神论者、劝人改变信仰等问题，与这个主题相符 |
| 2：通用讨论语言 | 21.82% | 文章含有表达观点、反驳和对话的语言，与这个主题的通用词特征相容 |
| 0、3、4 | 各约 0.31% | 比例很小，没有明显证据表明文章涉及航天或图像技术 |

注意：77.26% 是模型估计的主题混合比例，不是“这篇文章属于宗教类别的正确概率”。

### 示例文档的主题分布

该文档主要讨论信仰、无神论者以及讨论中的相互指责。

5 主题 LDA 模型估计：
- 主题 1“宗教与信仰讨论”约占 77.26%，与文章内容基本吻合。
- 主题 2“通用讨论语言”约占 21.82%，可能反映文章中的观点表达和争论用语。
- 其他三个主题各约占 0.31%，接近先验平滑带来的微小比例，
  不能据此认定文章涉及航天或图像技术。

这些数值是模型估计的主题混合比例，不是分类正确概率。
一篇文档可以混合多个主题，但每个主题是否有明确含义，
仍需结合关键词和正文判断。

## 4. 训练并检查 10 主题 LDA 模型

使用相同的词频矩阵，将主题数从 5 改为 10，
其他显式设置的参数保持一致。

输出每个主题的前 10 个词，观察：
- 是否出现更具体的讨论主题。
- 是否出现内容重复或难以解释的主题。
- 与 5 主题模型相比，主题是否更容易理解。

不同模型的主题编号不能直接对应：
10 主题模型中的主题 0，不一定对应 5 主题模型中的主题 0。

In [3]:
from sklearn.decomposition import LatentDirichletAllocation
from threadpoolctl import threadpool_limits

In [6]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from threadpoolctl import threadpool_limits

RANDOM_SEED = 42

categories = [
    "comp.graphics",
    "sci.space",
    "alt.atheism",
    "talk.religion.misc",
]

newsgroups = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=RANDOM_SEED,
)

text_df = pd.DataFrame({
    "text": newsgroups.data,
    "label": newsgroups.target,
})

label_to_category = dict(enumerate(newsgroups.target_names))
text_df["category"] = text_df["label"].map(label_to_category)

empty_mask = text_df["text"].str.strip().eq("")

clean_text_df = (
    text_df.loc[~empty_mask]
    .copy()
    .reset_index(names="original_index")
)

documents = clean_text_df["text"].tolist()
true_labels = clean_text_df["label"].to_numpy()

count_vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=2,
    max_df=0.95,
    max_features=5000,
)

X_counts = count_vectorizer.fit_transform(documents)
count_feature_names = count_vectorizer.get_feature_names_out()

retained_word_counts = np.asarray(X_counts.sum(axis=1)).ravel()
zero_count_mask = retained_word_counts == 0


def get_topic_words(model, feature_names, n_top_words=10):
    rows = []

    for topic_index, topic_weights in enumerate(model.components_):
        top_indices = topic_weights.argsort()[-n_top_words:][::-1]

        rows.append({
            "Topic": topic_index,
            "Top words": ", ".join(feature_names[top_indices]),
        })

    return pd.DataFrame(rows)


print("Random seed:", RANDOM_SEED)
print("Count matrix shape:", X_counts.shape)
print("Zero-count documents:", int(zero_count_mask.sum()))
print("Preparation completed.")

Random seed: 42
Count matrix shape: (3295, 5000)
Zero-count documents: 15
Preparation completed.


In [7]:
lda_10 = LatentDirichletAllocation(
    n_components=10,
    learning_method="batch",
    max_iter=20,
    doc_topic_prior=0.1,
    topic_word_prior=0.1,
    evaluate_every=-1,
    random_state=RANDOM_SEED,
    n_jobs=1,
    verbose=1,
)

print("Training LDA with 10 topics...")

with threadpool_limits(limits=1):
    doc_topic_10 = lda_10.fit_transform(X_counts)

print("LDA training completed.")
print("Training iterations:", lda_10.n_iter_)
print("Document-topic matrix shape:", doc_topic_10.shape)
print("Topic-word parameter shape:", lda_10.components_.shape)
print("First 5 document-topic row sums:", doc_topic_10[:5].sum(axis=1))

topics_10_df = get_topic_words(lda_10, count_feature_names)

with pd.option_context("display.max_colwidth", None):
    display(topics_10_df)

Training LDA with 10 topics...
iteration: 1 of max_iter: 20
iteration: 2 of max_iter: 20
iteration: 3 of max_iter: 20
iteration: 4 of max_iter: 20
iteration: 5 of max_iter: 20
iteration: 6 of max_iter: 20
iteration: 7 of max_iter: 20
iteration: 8 of max_iter: 20
iteration: 9 of max_iter: 20
iteration: 10 of max_iter: 20
iteration: 11 of max_iter: 20
iteration: 12 of max_iter: 20
iteration: 13 of max_iter: 20
iteration: 14 of max_iter: 20
iteration: 15 of max_iter: 20
iteration: 16 of max_iter: 20
iteration: 17 of max_iter: 20
iteration: 18 of max_iter: 20
iteration: 19 of max_iter: 20
iteration: 20 of max_iter: 20
LDA training completed.
Training iterations: 20
Document-topic matrix shape: (3295, 10)
Topic-word parameter shape: (10, 5000)
First 5 document-topic row sums: [1. 1. 1. 1. 1.]


,Topic,Top words
0,0,"space, nasa, launch, satellite, shuttle, research, program, new, technology, year"
1,1,"edu, graphics, ftp, pub, mail, send, available, com, data, 3d"
2,2,"theory, universe, gamma, larson, point, energy, problem, physical, like, does"
3,3,"jpeg, image, file, gif, format, images, color, files, bit, use"
4,4,"image, data, software, graphics, program, processing, video, display, images, using"
5,5,"earth, spacecraft, venus, planet, solar, mars, space, surface, probe, orbit"
6,6,"like, just, think, space, don, time, moon, years, know, people"
7,7,"know, just, people, don, like, book, think, read, ve, good"
8,8,"god, people, jesus, don, say, does, think, believe, just, bible"
9,9,"gopher, conference, station, search, judas, mr, software, den, p2, greek"


### 10 主题模型的关键词观察

与 5 主题模型相比，10 主题模型呈现出更细的部分讨论方向：

- 航天内容中，主题 0 偏向航天计划与技术，
  主题 5 偏向行星与深空探测。
- 图形学内容中，主题 1 可能涉及资源与网络交流，
  主题 3 偏向图像格式与编码，
  主题 4 偏向图像处理与软件工具。
- 主题 8 具有较明确的宗教与信仰含义。

但是，并非所有主题都容易解释：
- 主题 6 混合了太空词语与通用讨论用词。
- 主题 7 主要包含通用讨论与阅读用语。
- 主题 9 的关键词较混杂，暂时无法确定一致含义。

增加主题数可以使部分主题更加具体，
也可能产生重叠或难以解释的主题，不能认为主题越多越好。

以上名称仅依据前十个关键词作出，仍需结合代表文档检查。
不同模型的主题编号不能直接对应。

## 5. 训练并检查 15 主题 LDA 模型

沿用同一个词频矩阵和其他显式参数，将主题数设置为 15。

输出每个主题的前 10 个词，观察主题是否更加具体，
以及是否出现更多重复、混杂或难以解释的主题。

In [9]:
lda_15 = LatentDirichletAllocation(
    n_components=15,
    learning_method="batch",
    max_iter=20,
    doc_topic_prior=0.1,
    topic_word_prior=0.1,
    evaluate_every=-1,
    random_state=RANDOM_SEED,
    n_jobs=1,
    verbose=1,
)

print("Training LDA with 15 topics...")

with threadpool_limits(limits=1):
    doc_topic_15 = lda_15.fit_transform(X_counts)

print("LDA training completed.")
print("Training iterations:", lda_15.n_iter_)
print("Document-topic matrix shape:", doc_topic_15.shape)
print("Topic-word parameter shape:", lda_15.components_.shape)
print("First 5 document-topic row sums:", doc_topic_15[:5].sum(axis=1))

topics_15_df = get_topic_words(lda_15, count_feature_names)

with pd.option_context("display.max_colwidth", None):
    display(topics_15_df)

Training LDA with 15 topics...
iteration: 1 of max_iter: 20
iteration: 2 of max_iter: 20
iteration: 3 of max_iter: 20
iteration: 4 of max_iter: 20
iteration: 5 of max_iter: 20
iteration: 6 of max_iter: 20
iteration: 7 of max_iter: 20
iteration: 8 of max_iter: 20
iteration: 9 of max_iter: 20
iteration: 10 of max_iter: 20
iteration: 11 of max_iter: 20
iteration: 12 of max_iter: 20
iteration: 13 of max_iter: 20
iteration: 14 of max_iter: 20
iteration: 15 of max_iter: 20
iteration: 16 of max_iter: 20
iteration: 17 of max_iter: 20
iteration: 18 of max_iter: 20
iteration: 19 of max_iter: 20
iteration: 20 of max_iter: 20
LDA training completed.
Training iterations: 20
Document-topic matrix shape: (3295, 15)
Topic-word parameter shape: (15, 5000)
First 5 document-topic row sums: [1. 1. 1. 1. 1.]


,Topic,Top words
0,0,"space, launch, cost, shuttle, new, nasa, program, year, research, mission"
1,1,"edu, graphics, pub, mail, ftp, send, ray, 128, 3d, com"
2,2,"theory, universe, gamma, energy, larson, physical, light, matter, ray, time"
3,3,"data, available, graphics, ftp, edu, image, software, sgi, package, use"
4,4,"image, software, line, data, program, processing, images, points, display, analysis"
5,5,"islam, april, islamic, 1993, muslims, spacecraft, muslim, qur, rushdie, command"
6,6,"earth, moon, solar, planet, venus, orbit, surface, spacecraft, sky, atmosphere"
7,7,"thanks, know, like, does, help, looking, graphics, need, program, hi"
8,8,"god, people, does, don, say, believe, think, religion, evidence, point"
9,9,"gopher, software, search, mr, judas, den, p2, edu, p3, p1"


### 5、10、15 主题模型的初步比较

5 主题模型主要呈现较宽泛的讨论方向。
10 主题模型中，部分航天和图形学内容表现出更具体的侧重点。
15 主题模型中，宗教相关内容出现了更细的关键词组合，
例如信仰论证、基督教与圣经、伊斯兰相关讨论。

但是，主题数增加也伴随着问题：
- 部分主题的含义相近，例如多个航天主题。
- 部分主题主要体现通用讨论或求助用语。
- 部分主题的关键词混杂，难以概括一致含义。

因此，不能认为主题数越多越好。
目前只能根据关键词判断主题的可解释性，
还不能确定哪一个模型整体最好。

主题名称为人工暂定解释，需要结合代表文档检查。

In [10]:
#重点检查 15 主题模型中的主题 5 和主题 9
valid_positions = np.flatnonzero(~zero_count_mask)

for topic_id in [5, 9]:
    ranked_positions = valid_positions[
        np.argsort(doc_topic_15[valid_positions, topic_id])[::-1]
    ]

    print(f"\n========== Topic {topic_id} ==========")

    for document_index in ranked_positions[:2]:
        print("\nDocument position:", document_index)
        print(
            "Topic proportion:",
            round(float(doc_topic_15[document_index, topic_id]), 4),
        )
        print(
            "Retained word occurrences:",
            int(retained_word_counts[document_index]),
        )
        print("\nDocument preview:")
        print(documents[document_index][:1200])


========== Topic 5 ==========

Document position: 1218
Topic proportion: 0.9969
Retained word occurrences: 452

Document preview:
Forwarded from Neal Ausman, Galileo Mission Director

                                 GALILEO
                     MISSION DIRECTOR STATUS REPORT
                               POST-LAUNCH
                           April 23 - 29, 1993

SPACECRAFT

1.  On April 22 and 23, delta Differenced One-way Range (DOR) passes were
performed over DSS-14/63 (Goldstone/Madrid 70 meter antennas) and DSS-14/43
(Goldstone/Canberra 70 meter antennas), respectively.  Initial results 
indicate the delta DOR pass on April 22 was unsuccessful due to ground
station hardware problems but the one on April 23 was successfully performed.

2.  On April 23, a cruise science Memory Readout (MRO) was performed for the
Magnetometer (MAG) instrument.  Analysis indicates the data was received
properly.

3.  On April 23, the spare power relay contacts were commanded closed via the
spacecra

In [11]:
valid_positions = np.flatnonzero(~zero_count_mask)

for topic_id in [5, 9]:
    ranked_positions = valid_positions[
        np.argsort(doc_topic_15[valid_positions, topic_id])[::-1]
    ]

    print(f"\n========== Topic {topic_id} ==========")

    for document_index in ranked_positions[:2]:
        preview = " ".join(documents[document_index].split())[:400]

        print(
            f"\nDocument: {document_index} | "
            f"Proportion: {doc_topic_15[document_index, topic_id]:.4f} | "
            f"Word occurrences: {retained_word_counts[document_index]}"
        )
        print(preview)


========== Topic 5 ==========

Document: 1218 | Proportion: 0.9969 | Word occurrences: 452
Forwarded from Neal Ausman, Galileo Mission Director GALILEO MISSION DIRECTOR STATUS REPORT POST-LAUNCH April 23 - 29, 1993 SPACECRAFT 1. On April 22 and 23, delta Differenced One-way Range (DOR) passes were performed over DSS-14/63 (Goldstone/Madrid 70 meter antennas) and DSS-14/43 (Goldstone/Canberra 70 meter antennas), respectively. Initial results indicate the delta DOR pass on April 22 was un

Document: 908 | Proportion: 0.9943 | Word occurrences: 246
Forwarded from Neal Ausman, Galileo Mission Director GALILEO MISSION DIRECTOR STATUS REPORT POST-LAUNCH April 16 - 22, 1993 SPACECRAFT 1. On April 19, cruise science Memory Readouts (MROs) were performed for the Extreme Ultraviolet Spectrometer (EUV), Dust Detector (DDS), and Magnetometer (MAG) instruments. Preliminary analysis indicates the data was received properly. 2. On April 19, a Command De

========== Topic 9 ==========

Document: 1008

## 6. 代表文档检查

### 主题 5

主题关键词包含伊斯兰相关词语，同时混有 spacecraft、command 等航天词。

主题比例最高的两篇非零文档分别为：
- 文档 1218：Galileo 航天任务状态报告，主题比例约为 0.9969。
- 文档 908：另一篇 Galileo 航天任务状态报告，主题比例约为 0.9943。

两篇文章采用相似的报告格式。
这说明该主题明显包含航天任务报告，不能只命名为伊斯兰相关讨论。

结合关键词与代表文档，将其暂记为内容混杂的主题。
目前无法仅凭两篇文档确定混杂的具体原因。

### 主题 9

主题比例最高的两篇非零文档分别为：
- 文档 1008：包含图形学相关 C 代码，主题比例约为 0.9963。
- 文档 2116：介绍轨道参数、卫星追踪软件与资源，主题比例约为 0.9815。

这些代表文档涉及不同领域，关键词也较混杂，
因此暂时无法给出一致的语义名称。

### 检查结论

主题关键词需要结合代表文档解释。
较高的主题比例不代表人工命名正确，也不保证主题含义清楚。

15 主题模型呈现出部分更具体的讨论方向，
但也包含内容重叠或混杂的主题。
不能仅因主题更多，就认定模型效果更好。

## 7. 问题回答

### 1. 文档聚类与主题发现有什么区别？

本实验中的 K-means 将每篇文档分配到一个簇，
重点是把表示相近的文档放在一起。

LDA 学习每个主题的词语分布，
并将每篇文档表示为多个主题的混合比例，
重点是发现文档集合中的潜在讨论结构。

簇和主题都不一定与真实类别一一对应。

### 2. 一篇文档能否包含多个主题？

可以。LDA 为每篇文档估计一个主题比例向量，
各主题比例之和为 1。

例如，在 5 主题模型中，示例文档的主题 1 约占 77.26%，
主题 2 约占 21.82%，表现为宗教讨论与通用讨论语言的混合。

但非常小的非零比例可能主要来自先验平滑，
不能据此认定文章实际讨论了所有主题。

### 3. 没有类别标签时，主题模型为什么有用？

主题模型可以直接从文档的词项计数中学习，
不需要预先提供类别标签。

通过主题关键词和代表文档，
我们可以初步探索大量文本中有哪些讨论方向，
以及每篇文档如何混合这些主题。

但是，主题含义仍需要人工检查，
模型也可能产生含义不清楚的主题。